# 01 — Data Collection
Pulling historical home game attendance data for the New York Mets from the MLB Stats API.

Note: `requests` is a Python library that lets you fetch data from the internet in your code — basically the same thing as typing a URL in your browser, but done programmatically (through code).

In [ ]:
import pandas as pd
import numpy as np
import requests

print(f"pandas {pd.__version__} ✓")
print(f"numpy {np.__version__} ✓")
print("requests ✓")

## Pull Home Game Data
The MLB Stats API is free and official. We pull every regular season home game for the Mets across 4 seasons (2022-2025, post-COVID).

New York Mets team ID = 121.

Note: `game_pk` is the unique game ID assigned by MLB. PK stands for 'primary key' — a database term for a unique identifier. We'll use it later to link tables together.

In [ ]:
TEAM_ID = 121  # New York Mets
SEASONS = [2022, 2023, 2024, 2025]

def get_home_games(team_id, season):
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "season": season,
        "gameType": "R",
        "teamId": team_id,
        "hydrate": "linescore,team",
    }
    response = requests.get(url, params=params)
    data = response.json()

    games = []
    for date in data.get("dates", []):
        for game in date.get("games", []):
            home_team_id = game["teams"]["home"]["team"]["id"]
            if home_team_id == team_id:
                attendance = game.get("attendance") or game.get("teams", {}).get("home", {}).get("attendance")
                games.append({
                    "date": date["date"],
                    "season": season,
                    "opponent": game["teams"]["away"]["team"]["name"],
                    "home_score": game["teams"]["home"].get("score"),
                    "away_score": game["teams"]["away"].get("score"),
                    "attendance": attendance,
                    "status": game["status"]["detailedState"],
                    "game_pk": game["gamePk"]
                })
    return games

# Pull all seasons
all_games = []
for season in SEASONS:
    games = get_home_games(TEAM_ID, season)
    all_games.extend(games)
    print(f"{season}: {len(games)} home games found")

df = pd.DataFrame(all_games)
print(f"\nTotal: {len(df)} games")
print(f"Games with attendance data: {df['attendance'].notna().sum()}")
print(f"Games missing attendance: {df['attendance'].isna().sum()}")
df.head(10)

## Fetch Attendance via Individual Game Endpoints
If attendance is still showing as None above, we fetch it game-by-game using each game's unique game_pk.
This is slower but guaranteed to get the official recorded attendance for every completed game.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

def fetch_attendance(game_pk):
    try:
        url = "https://statsapi.mlb.com/api/v1/game/" + str(game_pk) + "/boxscore"
        data = requests.get(url, timeout=10).json()
        for item in data.get("info", []):
            if item.get("label") == "Att":
                val = re.sub(r"[^0-9]", "", item.get("value", ""))
                if val:
                    return game_pk, int(val)
    except:
        pass
    return game_pk, None

missing = df[df["attendance"].isna() & (df["status"] == "Final")]
print("Fetching attendance for", len(missing), "games in parallel...")

results = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_attendance, pk): pk for pk in missing["game_pk"]}
    for future in as_completed(futures):
        game_pk, att = future.result()
        results[game_pk] = att

for idx, row in missing.iterrows():
    df.at[idx, "attendance"] = results.get(row["game_pk"])

filled = sum(1 for v in results.values() if v is not None)
print("Successfully fetched:", filled, "/", len(missing))
print("Sample attendance values:")
print(df[df["attendance"].notna()][["date", "opponent", "attendance"]].head(10))

## Remove Neutral Site Games
Some games are listed as Mets home games in the API but were played at neutral sites (London Series, etc.). These have different attendance dynamics and should be excluded from our Citi Field model.

Removed:
- 2024-06-08, 2024-06-09 — London Series at London Stadium
- 2025-08-17 — MLB Little League Classic at Journey Bank Ballpark

In [10]:
neutral_site_dates = ["2024-06-08", "2024-06-09", "2025-08-17"]
df = df[~df["date"].isin(neutral_site_dates)]
print(f"After removing neutral site games: {len(df)} games remaining")

After removing neutral site games: 337 games remaining


## Save Training Data to CSV
Save 2022-2025 data to the data/ folder. This is our training data — we never modify this file.

Note: Citi Field official capacity is 41,922. All attendance figures represent tickets sold, not actual turnstile attendance.

In [12]:
df.to_csv("../data/mets_games_raw.csv", index=False)
print("Saved to data/mets_games_raw.csv")
print(f"\nColumns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nAttendance summary:")
print(df['attendance'].astype(float).describe())

Saved to data/mets_games_raw.csv

Columns: ['date', 'season', 'opponent', 'home_score', 'away_score', 'attendance', 'status', 'game_pk']
Date range: 2022-04-15 to 2025-09-21

Attendance summary:
count      311.000000
mean     33875.961415
std       6899.990787
min      15020.000000
25%      28514.500000
50%      34876.000000
75%      40022.500000
max      44152.000000
Name: attendance, dtype: float64


## Assess Outliers
Check for any remaining low-attendance games and assess whether they're legitimate.

In [11]:
print("Training data (2022-2025) games under 20k:")
print(df[df['attendance'] < 20000][['date', 'opponent', 'attendance']])

Training data (2022-2025) games under 20k:
           date            opponent attendance
176  2024-04-01      Detroit Tigers      16853
180  2024-04-04      Detroit Tigers      15020
181  2024-04-12  Kansas City Royals      18822
184  2024-04-15  Pittsburgh Pirates      18266
185  2024-04-16  Pittsburgh Pirates      18398
186  2024-04-17  Pittsburgh Pirates      18092
196  2024-05-12      Atlanta Braves      18944
212  2024-06-12       Miami Marlins      19803


## Pull 2026 Data (Prediction & Validation)
We keep 2026 completely separate from our training data.
- Games already played in 2026 let us check how accurate our model is against real results.
- Future 2026 games are genuine forecasts — no answer key yet.

In [ ]:
games_2026 = get_home_games(TEAM_ID, 2026)
df_2026 = pd.DataFrame(games_2026)

missing_2026 = df_2026[df_2026["attendance"].isna() & (df_2026["status"] == "Final")]
print("Fetching attendance for", len(missing_2026), "completed 2026 games...")

results_2026 = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_attendance, pk): pk for pk in missing_2026["game_pk"]}
    for future in as_completed(futures):
        game_pk, att = future.result()
        results_2026[game_pk] = att

for idx, row in missing_2026.iterrows():
    df_2026.at[idx, "attendance"] = results_2026.get(row["game_pk"])

played = df_2026[df_2026["status"] == "Final"]
upcoming = df_2026[df_2026["status"] != "Final"]
print("2026 home games:", len(df_2026))
print("  Already played:", len(played))
print("  Not yet played:", len(upcoming))
df_2026.head(10)

In [ ]:
df_2026.to_csv("../data/mets_2026_prediction.csv", index=False)
print("Saved to data/mets_2026_prediction.csv")